# Logistic Regression — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/machine-learning/2-1/logistic-regression.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/machine-learning/2-1/logistic-regression.ipynb)

> Companion to the **Logistic Regression** article. We build logistic regression **from scratch in pure numpy** —
> squash a linear score through the **sigmoid**, fit by **gradient descent on log-loss**, and watch the punchline:
> **`exp(β)` is an odds ratio**, the same number a 2×2 table gives you.

## What you'll build

1. The **sigmoid** and **log-loss**, coded by hand.
2. A tiny **pass/fail vs hours-studied** dataset (the article's example).
3. **Gradient descent** on log-loss — and watch the loss fall every step.
4. The **sigmoid curve** of learned probabilities.
5. The bridge: **`exp(β₁)` = the odds ratio**, tied back to the 2×2 table.

**No API key, no sklearn required** — pure numpy, so it runs anywhere top-to-bottom.

### Setup (only numpy needed)

In [1]:
# pure numpy — nothing else required
# %pip install -q numpy   # uncomment on a bare environment
import numpy as np
rng = np.random.default_rng(0)
print("ready — numpy", np.__version__)

ready — numpy 2.2.6


## The two building blocks: sigmoid and log-loss

The **sigmoid** squashes any real score `z` into a probability in `(0, 1)`:
`σ(z) = 1/(1+e^−z)`. **Log-loss** (binary cross-entropy) measures how wrong our
probabilities are — it's what we minimize.

In [2]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def log_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)          # avoid log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# sanity check: sigma(0) = 0.5 (a coin flip), and the curve is monotone
for z in [-2, -0.5, 0, 0.5, 2]:
    print(f"  sigmoid({z:>4}) = {sigmoid(z):.3f}")

  sigmoid(  -2) = 0.119
  sigmoid(-0.5) = 0.378
  sigmoid(   0) = 0.500
  sigmoid( 0.5) = 0.622
  sigmoid(   2) = 0.881


## The dataset: does studying predict passing?

We generate data from a **known** model — `P(pass) = σ(−4 + 1.5·hours)` — so we can
later check that the fit *recovers* those coefficients. One feature (**hours**), a
noisy 0/1 **passed** label. We prepend a column of 1s so the intercept β₀ is just
another coefficient.

In [3]:
TRUE_B0, TRUE_B1 = -4.0, 1.5          # the truth we hope to recover
n = 200
hours = rng.uniform(0, 6, size=n)                     # hours studied
p_true = sigmoid(TRUE_B0 + TRUE_B1 * hours)
passed = (rng.random(n) < p_true).astype(int)         # noisy 0/1 labels
X = np.column_stack([np.ones(n), hours])              # [1, hours] -> intercept + slope
y = passed.astype(float)

print(f"generated {n} students from P(pass) = sigmoid({TRUE_B0} + {TRUE_B1}*hours)")
print("passed overall:", int(passed.sum()), "of", n)
print("design matrix X (first 3 rows):")
print(X[:3])

generated 200 students from P(pass) = sigmoid(-4.0 + 1.5*hours)
passed overall: 110 of 200
design matrix X (first 3 rows):
[[1.         3.82177012]
 [1.         1.61872028]
 [1.         0.24584114]]


## Fit by gradient descent on log-loss

No closed form — we descend. The gradient of log-loss is beautifully simple:
`∂L/∂βⱼ = (1/n) Σ (p − y)·xⱼ` — *prediction minus truth, times the feature*.
Because log-loss is **convex**, we're guaranteed to reach the global minimum.

In [4]:
beta = np.zeros(X.shape[1])
eta, epochs = 0.3, 20000
history = []
for epoch in range(epochs):
    p = sigmoid(X @ beta)                 # predicted probabilities
    grad = X.T @ (p - y) / len(y)         # dL/dbeta = mean((p - y) . x)
    beta -= eta * grad                    # step downhill
    if epoch % 4000 == 0:
        history.append((epoch, log_loss(y, sigmoid(X @ beta))))

b0, b1 = beta
print(f"learned coefficients:  beta0 = {b0:.3f}   beta1 = {b1:.3f}")
print("\nlog-loss DECREASING as we learn:")
for epoch, L in history:
    print(f"  epoch {epoch:6d}   log-loss = {L:.4f}")

learned coefficients:  beta0 = -4.867   beta1 = 1.623

log-loss DECREASING as we learn:
  epoch      0   log-loss = 0.5844
  epoch   4000   log-loss = 0.3109
  epoch   8000   log-loss = 0.3109
  epoch  12000   log-loss = 0.3109
  epoch  16000   log-loss = 0.3109


## The learned sigmoid curve

Feed hours through `z = β₀ + β₁·hours` and squash. Notice the S-shape: near 0
for few hours, near 1 for many, and crossing 0.5 at the **decision boundary**
`hours = −β₀/β₁`.

In [5]:
print("hours -> z -> P(pass):")
for h in [1, 2, 3, 4, 5]:
    z = b0 + b1 * h
    print(f"  {h} hours -> z = {z:6.2f}   p = {sigmoid(z):.3f}")

boundary = -b0 / b1
print(f"\ndecision boundary (p = 0.5) at hours = -beta0/beta1 = {boundary:.2f}")

# a compact ASCII sigmoid over a fine grid
grid = np.linspace(0, 6, 25)
probs = sigmoid(b0 + b1 * grid)
print("\nsigmoid curve (each row: hours | bar ~ P(pass)):")
for h, pr in zip(grid[::2], probs[::2]):
    print(f"  {h:4.1f} | " + "#" * int(round(pr * 40)) + f"  {pr:.2f}")

hours -> z -> P(pass):
  1 hours -> z =  -3.24   p = 0.038
  2 hours -> z =  -1.62   p = 0.165
  3 hours -> z =   0.00   p = 0.500
  4 hours -> z =   1.62   p = 0.835
  5 hours -> z =   3.25   p = 0.963

decision boundary (p = 0.5) at hours = -beta0/beta1 = 3.00

sigmoid curve (each row: hours | bar ~ P(pass)):
   0.0 |   0.01
   0.5 | #  0.02
   1.0 | ##  0.04
   1.5 | ###  0.08
   2.0 | #######  0.16
   2.5 | ############  0.31
   3.0 | ####################  0.50
   3.5 | ############################  0.69
   4.0 | #################################  0.84
   4.5 | #####################################  0.92
   5.0 | #######################################  0.96
   5.5 | #######################################  0.98
   6.0 | ########################################  0.99


## The bridge: `exp(β₁)` is an ODDS RATIO

Raise `e` to the log-odds line and a one-unit rise in a feature **multiplies the
odds** by `exp(β)`. That multiplier is exactly an **odds ratio** — the same
`ad/bc` you'd get from a 2×2 table. Let's confirm it directly from the data.

In [6]:
OR = np.exp(b1)
print(f"exp(beta1) = {OR:.3f}   <-- ODDS RATIO")
print(f"each extra hour multiplies the ODDS of passing by {OR:.2f}x\n")

# verify: odds ratio between studying h+1 vs h hours == exp(beta1), for ANY h
for h in [1.0, 2.5, 4.0]:
    p_lo = sigmoid(b0 + b1 * h)
    p_hi = sigmoid(b0 + b1 * (h + 1))
    odds_lo = p_lo / (1 - p_lo)
    odds_hi = p_hi / (1 - p_hi)
    print(f"  {h:>3} -> {h+1:>3} hrs:  odds {odds_lo:6.3f} -> {odds_hi:6.3f}   "
          f"ratio = {odds_hi/odds_lo:.3f}")
print(f"\nevery ratio equals exp(beta1) = {OR:.3f}  -> the effect on the odds is CONSTANT")

exp(beta1) = 5.066   <-- ODDS RATIO
each extra hour multiplies the ODDS of passing by 5.07x

  1.0 -> 2.0 hrs:  odds  0.039 ->  0.198   ratio = 5.066
  2.5 -> 3.5 hrs:  odds  0.445 ->  2.253   ratio = 5.066
  4.0 -> 5.0 hrs:  odds  5.070 -> 25.687   ratio = 5.066

every ratio equals exp(beta1) = 5.066  -> the effect on the odds is CONSTANT


### Tie it back to the 2×2 table

For a single **binary** feature, the logistic OR `exp(β)` equals the table's `ad/bc`.
Here's the check: split hours into low/high and compare the two ways of getting an OR.

In [7]:
# binary exposure: studied a lot (>= 3 hrs) vs not, against passing
high = (hours >= 3.0).astype(int)
a = int(np.sum((high == 1) & (passed == 1)))   # high & pass
b = int(np.sum((high == 1) & (passed == 0)))   # high & fail
c = int(np.sum((high == 0) & (passed == 1)))   # low  & pass
d = int(np.sum((high == 0) & (passed == 0)))   # low  & fail
print(f"2x2 cells:  a={a} b={b} c={c} d={d}")

# Haldane-Anscombe +0.5 only if a zero cell, then OR = ad/bc
aa, bb, cc, dd = (a, b, c, d) if min(a, b, c, d) else (a+.5, b+.5, c+.5, d+.5)
OR_table = (aa * dd) / (bb * cc)
print(f"odds ratio from 2x2 table (ad/bc) = {OR_table:.3f}")

# fit a logistic model on the SAME binary feature -> exp(beta)
Xb = np.column_stack([np.ones(n), high.astype(float)])
beta_b = np.zeros(2)
for _ in range(60000):
    pb = sigmoid(Xb @ beta_b)
    beta_b -= 0.3 * (Xb.T @ (pb - y) / n)
print(f"odds ratio from logistic exp(beta) = {np.exp(beta_b[1]):.3f}")
print("\nsame number two ways -> logistic regression's exp(beta) IS the 2x2 odds ratio.")

2x2 cells:  a=99 b=13 c=11 d=77
odds ratio from 2x2 table (ad/bc) = 53.308


odds ratio from logistic exp(beta) = 53.308

same number two ways -> logistic regression's exp(beta) IS the 2x2 odds ratio.


## The sklearn one-liner (optional)

The from-scratch loop above is the whole idea. In practice you'd call a library —
same model, batteries included:

```python
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(C=1e6).fit(hours.reshape(-1, 1), passed)
print(clf.intercept_, clf.coef_, np.exp(clf.coef_))  # exp(coef) = odds ratio
```

## Recap

- **Sigmoid** turns a linear score into a probability; **log-loss** measures how wrong it is.
- **Gradient descent** on the convex log-loss recovers the coefficients — the loss fell every step.
- The model is **linear in the log-odds**, so **`exp(β)` is an odds ratio**: a constant multiplicative effect on the odds.
- For a binary feature, that `exp(β)` **equals the 2×2 table's `ad/bc`** — logistic regression's natural output is the odds ratio.

← Back to the **Logistic Regression** article for the full derivation, diagrams, and interview Q&A.